# Pet segmentation: training on a free Colab GPU

The repository's experiments were run on a laptop CPU at 160 px for 10 epochs. This notebook reproduces them on a free Colab T4 GPU, and can also run the larger configuration (224 px, more epochs) that the CPU budget did not allow.

Runtime -> Change runtime type -> T4 GPU.

In [ ]:
# Replace with your fork's URL
!git clone https://github.com/<your-user>/pet-segmentation.git
%cd pet-segmentation
!pip install -q -r requirements.txt

In [ ]:
# ~800 MB download, writes a compact copy to data/processed
!python scripts/prepare_data.py --delete-archives
!python scripts/explore_data.py

In [ ]:
import torch; print(torch.cuda.get_device_name(0))
# All 10 runs from the report (architectures, ablations, final) (same configs as on CPU)
!python scripts/run_experiments.py

In [ ]:
# Optional: larger run that did not fit the CPU budget
!python -m petseg.train configs/lraspp_mbv3.yaml --name lraspp_mbv3_224_e30 --size 224 --epochs 30

In [ ]:
# Test-set evaluation, optimisation study, and the shipped ONNX models
!python -m petseg.evaluate runs/final/best.pt
!python scripts/optimization_study.py --deploy final
!python -m petseg.export runs/final/best.pt --out models/petseg --size 224 --calib percentile

In [ ]:
# Download the trained model
from google.colab import files
files.download('runs/final/best.pt')
files.download('models/petseg.onnx')